# Stage 5 — Descriptive Trend Analysis


## Setup

In [ ]:
import os, time, gc
import numpy as np, pandas as pd, pyarrow as pa
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

ROOT   = r"D:\Code\projects\Consumer_complaint_analytics"
CLEAN  = os.path.join(ROOT, "data", "processed", "complaints_clean.parquet")
ML     = os.path.join(ROOT, "data", "processed", "complaints_ml.parquet")
CHARTS = os.path.join(ROOT, "charts")
REPORTS= os.path.join(ROOT, "reports")
LABELS = ["Credit reporting", "Debt collection", "Mortgage", "Credit card"]
COLORS = dict(zip(LABELS, sns.color_palette("tab10", 4)))
LS     = pd.ArrowDtype(pa.large_string())

def finish(fig, fname, caption):
    '''Add an italic caption under the figure, save to charts/, and display inline.'''
    fig.text(0.5, -0.02, caption, ha="center", va="top", fontsize=8.5, style="italic", wrap=True)
    fig.savefig(os.path.join(CHARTS, fname), dpi=130, bbox_inches="tight")
    plt.show()

def thousands(ax):
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
print("Setup ready. Charts ->", CHARTS)

### Load the structured table (all complaints)



In [ ]:
t = time.time()
COLS = ["category", "year", "year_month", "Company",
        "Company response to consumer", "Timely response?", "has_narrative"]
df = pd.read_parquet(CLEAN, columns=COLS)
df["category"]   = df["category"].astype("string").astype("category")
df["year"]       = df["year"].astype("int64")
df["year_month"] = df["year_month"].astype("string")
df["month"]      = df["year_month"].str.slice(5, 7).astype("int64")
print(f"Loaded {len(df):,} complaints in {time.time()-t:.0f}s")
df[["category","year","year_month","month","Company response to consumer"]].head(3)

## 1. Category shares — how the four categories split



In [ ]:
counts = df["category"].value_counts().reindex(LABELS)
shares = (counts / counts.sum() * 100)

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(LABELS, counts.values, color=[COLORS[c] for c in LABELS])
thousands(ax); ax.set_ylabel("Complaints (2011–2026)")
ax.set_title("Category shares of the four-category complaint set")
for b, c, s in zip(bars, counts.values, shares.values):
    ax.text(b.get_x()+b.get_width()/2, c, f"{c:,.0f}\n({s:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.margins(y=0.18)
finish(fig, "stage5_01_category_shares.png",
       "Figure 1. Of 15.26M complaints across the four categories, credit reporting dominates at 86.2%; "
       "debt collection 7.5%, credit card 3.4%, mortgage 3.0%. Counts include all complaints (templated "
       "re-filings included).")
shares.round(2)

## 2. Volume over time — the credit-reporting surge 

In [ ]:
by_year = (df.groupby(["year","category"], observed=True).size()
             .unstack("category").reindex(columns=LABELS).sort_index())

fig, ax = plt.subplots(figsize=(9, 5))
for c in LABELS:
    ax.plot(by_year.index, by_year[c], marker="o", ms=4, label=c, color=COLORS[c])
thousands(ax); ax.set_xlabel("Year"); ax.set_ylabel("Complaints"); ax.legend(title="Category")
ax.set_title("Complaints per category per year (all complaints)")
ax.axvspan(2025.5, 2026.5, color="grey", alpha=0.12)
ax.text(2026, by_year["Credit reporting"].max()*0.5, "2026\n(partial)", ha="center", fontsize=8, color="grey")
ax.annotate(f"604k (2022)\n-> 4.81M (2025)", xy=(2025, by_year.loc[2025,"Credit reporting"]),
            xytext=(2021.2, by_year["Credit reporting"].max()*0.85),
            arrowprops=dict(arrowstyle="->", color="black"), fontsize=9)
finish(fig, "stage5_02a_volume_by_year.png",
       "Figure 2a. Credit-reporting complaints surged ~8x in three years (604,181 in 2022 to 4,810,346 "
       "in 2025), dwarfing the other categories. 2026 is a partial year. The smaller categories are "
       "near-flat on this shared axis and are shown separately in Figure 2b.")
by_year

The surge is so large that the other three categories look flat above. To read them, we plot the
**three smaller categories on their own axis**.

In [ ]:
small = ["Debt collection", "Mortgage", "Credit card"]
fig, ax = plt.subplots(figsize=(9, 4.5))
for c in small:
    ax.plot(by_year.index, by_year[c], marker="o", ms=4, label=c, color=COLORS[c])
thousands(ax); ax.set_xlabel("Year"); ax.set_ylabel("Complaints"); ax.legend(title="Category")
ax.set_title("The three smaller categories (own axis, all complaints)")
ax.axvspan(2025.5, 2026.5, color="grey", alpha=0.12)
finish(fig, "stage5_02b_volume_by_year_small3.png",
       "Figure 2b. With credit reporting removed, debt collection rose gradually then dipped recently, "
       "credit card grew modestly, and mortgage drifted down. None show anything like the credit-"
       "reporting explosion. 2026 partial.")

For finer texture, the same volumes **by month**.

In [ ]:
by_month = (df.groupby(["year_month","category"], observed=True).size()
              .unstack("category").reindex(columns=LABELS).sort_index())
x = pd.to_datetime(by_month.index + "-01")
fig, ax = plt.subplots(figsize=(10, 4.5))
for c in LABELS:
    ax.plot(x, by_month[c], label=c, color=COLORS[c], lw=1.2)
thousands(ax); ax.set_xlabel("Month"); ax.set_ylabel("Complaints / month"); ax.legend(title="Category")
ax.set_title("Monthly complaint volume by category (all complaints)")
finish(fig, "stage5_02c_volume_by_month.png",
       "Figure 2c. Monthly view of the same data: credit reporting accelerates sharply from ~2022 "
       "onward, exceeding 600,000 complaints in single recent months. Last point is mid-2026 (partial).")

## 3. Check for Templated-share trend



In [ ]:
del df; gc.collect()   # free the structured table before the text step (memory discipline)

t = time.time()
ml = pd.read_parquet(ML, columns=["narrative_clean", "Date received"],
                     filters=[("category", "==", "Credit reporting")])
ml["narrative_clean"] = ml["narrative_clean"].astype(LS)
ml["year"] = ml["Date received"].dt.year.astype("int64")
print(f"Loaded {len(ml):,} credit-reporting narratives in {time.time()-t:.0f}s")

# GLOBAL: code each distinct text across all years; templated if it appears >= 2x in total
g_codes, _ = pd.factorize(ml["narrative_clean"])
ml["tmpl_global"] = np.bincount(g_codes)[g_codes] >= 2

# WITHIN-YEAR: repeat the coding separately inside each year
def flag_within(s):
    c, _ = pd.factorize(s); return pd.Series(np.bincount(c)[c] >= 2, index=s.index)
ml["tmpl_winyr"] = ml.groupby("year", group_keys=False)["narrative_clean"].apply(flag_within)

g = ml.groupby("year")
tmpl = pd.DataFrame({"cr_narratives": g.size(),
                     "templated_global":  g["tmpl_global"].sum().astype(int),
                     "templated_winyear": g["tmpl_winyr"].sum().astype(int)})
tmpl["share_global_%"]  = (tmpl["templated_global"]  / tmpl["cr_narratives"] * 100).round(1)
tmpl["share_winyear_%"] = (tmpl["templated_winyear"] / tmpl["cr_narratives"] * 100).round(1)
del ml; gc.collect()
tmpl

⚠️ **2026 is unreliable here and is excluded from the charts below.** Published narratives lag the
complaint by months (consent + redaction), so 2026 shows only ~3,200 CR narratives against ~3.3M CR
complaints — far too few to trust. We plot 2015–2025.

In [ ]:
tp = tmpl.loc[2015:2025]

fig, axes = plt.subplots(2, 1, figsize=(9, 7.5), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1]})
# top: percentage share, both definitions
ax = axes[0]
ax.plot(tp.index, tp["share_winyear_%"], marker="o", color="crimson", label="Within-year definition")
ax.plot(tp.index, tp["share_global_%"],  marker="s", ls="--", color="grey", label="Global definition")
ax.set_ylabel("% of CR narratives\nthat are templated"); ax.set_ylim(0, 100); ax.legend()
ax.set_title("Templated share of credit-reporting narratives, by year")
for yr in (2015, 2025):
    ax.annotate(f"{tp.loc[yr,'share_winyear_%']:.0f}%", (yr, tp.loc[yr,"share_winyear_%"]),
                textcoords="offset points", xytext=(0,8), ha="center", fontsize=9, color="crimson")
# bottom: absolute templated volume
ax = axes[1]
ax.bar(tp.index, tp["templated_winyear"], color="crimson", alpha=0.8)
thousands(ax); ax.set_ylabel("Templated CR narratives\n(count, within-year)"); ax.set_xlabel("Year")
finish(fig, "stage5_03b_cr_templated_share.png",
       "Figure 3b. The templated share of credit-reporting narratives rose from ~11% (2015) to ~77% "
       "(2025); the within-year and global definitions almost coincide, so the growth is NOT an artifact "
       "of counting duplicates across years. Absolute templated volume (lower panel) grew from ~1,100 to "
       "~705,000 narratives/year. 2026 excluded (incomplete narrative publication).")

And the composition of each year's CR narratives — **templated vs distinct** — in absolute terms.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
distinct = tp["cr_narratives"] - tp["templated_winyear"]
ax.bar(tp.index, tp["templated_winyear"], label="Templated (within-year duplicate)", color="crimson", alpha=0.85)
ax.bar(tp.index, distinct, bottom=tp["templated_winyear"], label="Distinct narrative", color="steelblue", alpha=0.85)
thousands(ax); ax.set_xlabel("Year"); ax.set_ylabel("CR narratives"); ax.legend()
ax.set_title("Credit-reporting narratives each year: templated vs distinct")
finish(fig, "stage5_03a_cr_templated_counts.png",
       "Figure 3a. Both the templated and distinct portions grew, but the templated (red) portion grew "
       "far faster, coming to dominate recent years. By 2025 roughly three in four CR narratives are "
       "mass-produced boilerplate. 2026 excluded (incomplete).")

**Reading of Section 3.** The within-year and global lines are within ~0.3 points of each other
every year, so the rise is real: the templates are filed in bulk *within* each
year. Both the *proportion* (11% → 77%) and the *raw scale* (~1,100 → ~705,000 per year) of templated
filings climbed steeply — a strong signal that automated/templated "dispute-mill" filing, not a
broadening of distinct consumer grievances, drives much of the credit-reporting surge.

### Reload the structured table

We freed it for the text step; reload the columns the remaining sections need.

In [ ]:
df = pd.read_parquet(CLEAN, columns=COLS)
df["category"]   = df["category"].astype("string").astype("category")
df["year"]       = df["year"].astype("int64")
df["year_month"] = df["year_month"].astype("string")
df["month"]      = df["year_month"].str.slice(5, 7).astype("int64")
print(f"Reloaded {len(df):,} complaints")

## 4. Seasonality — repeating within-year calendar patterns



In [ ]:
ym = df.groupby(["year","month","category"], observed=True).size().rename("n").reset_index()
yt = df.groupby(["year","category"], observed=True).size().rename("yt").reset_index()
ym = ym.merge(yt, on=["year","category"]); ym["frac"] = ym["n"] / ym["yt"]
complete = list(range(2012, 2026))
prof = (ym[ym["year"].isin(complete)].groupby(["category","month"], observed=True)["frac"].mean()
          .unstack("category").reindex(columns=LABELS)) * 100

fig, ax = plt.subplots(figsize=(8.5, 4.6))
for c in LABELS:
    ax.plot(prof.index, prof[c], marker="o", ms=4, label=c, color=COLORS[c])
ax.set_xticks(range(1,13)); ax.set_xlabel("Calendar month")
ax.set_ylabel("Avg % of that year's complaints"); ax.legend(title="Category")
ax.set_title("Average within-year monthly profile (complete years 2012–2025)")
ax.axhline(100/12, color="grey", ls=":", lw=1)
finish(fig, "stage5_04b_month_profile.png",
       "Figure 4b. Dotted line = an even 8.33%/month (no seasonality). Credit reporting ramps up Jan->Dec, "
       "but that reflects its steep year-on-year growth leaking in, not a true season. Mortgage and debt "
       "collection are closer to flat with mild mid-year lifts. Formal trend/season separation is Stage 6.")
prof.round(2)

A **heatmap** of raw monthly credit-reporting volume (year × month) shows both the rising trend
(down the rows) and any repeating monthly pattern (across columns) at once.

In [ ]:
cr_hm = (df[df["category"]=="Credit reporting"].groupby(["year","month"], observed=True).size()
           .unstack("month").reindex(index=range(2012,2027)))
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(cr_hm/1000, cmap="rocket_r", ax=ax, cbar_kws={"label":"complaints (thousands)"})
ax.set_xlabel("Month"); ax.set_ylabel("Year"); ax.set_title("Credit-reporting complaints by year × month")
finish(fig, "stage5_04a_seasonality_heatmap.png",
       "Figure 4a. Colour = monthly credit-reporting volume. The dominant signal is the trend (rows get "
       "hotter over time); 2026 is partial (blank later months). No strong fixed-month cycle stands out "
       "above the trend — consistent with Figure 4b. Stage 6 will decompose trend vs season formally.")

## 5. Company breakdown — check which company attracts the most complaints



In [ ]:
top = df["Company"].astype("string").value_counts().head(15)[::-1]
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top.index, top.values, color="slateblue")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
ax.set_xlabel("Complaints (2011–2026)"); ax.set_title("Top 15 companies by complaint volume")
finish(fig, "stage5_05a_top_companies_overall.png",
       "Figure 5a. The three national credit bureaus (TransUnion 4.37M, Equifax 4.29M, Experian 3.88M) "
       "receive an order of magnitude more complaints than any other firm — unsurprising given the "
       "credit-reporting surge. VOLUME, not rate: larger firms draw more complaints by size alone.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, c in zip(axes.ravel(), LABELS):
    t5 = df.loc[df["category"]==c, "Company"].astype("string").value_counts().head(5)[::-1]
    ax.barh(t5.index, t5.values, color=COLORS[c])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
    ax.set_title(c, fontsize=10); ax.tick_params(labelsize=8)
fig.suptitle("Top 5 companies within each category", y=1.0)
fig.tight_layout()
finish(fig, "stage5_05b_top_companies_by_category.png",
       "Figure 5b. Within each category the leaders differ: credit bureaus in credit reporting; debt "
       "buyers/agencies in debt collection; large banks in mortgage and credit card. Raw volume, not "
       "rate (no market-share normalisation available).")

## 6. Resolution outcomes — how complaints end

Two columns describe outcomes. **`Company response to consumer`** is how the company closed it — e.g.
*Closed with explanation* (a reply, no money or action), *Closed with non-monetary relief* (a fix such as
correcting a credit file, no cash), *Closed with monetary relief* (money back), or *In progress*.
**`Timely response?`** is whether the company replied within the regulator's deadline (Yes/No). We look at
the overall mix, then whether it differs **by category**, by **timeliness**, and by whether the complaint
**carried a narrative** (`has_narrative`).

In [ ]:
resp = (df["Company response to consumer"].astype("string")
          .fillna("(missing)").value_counts())
labels = [str(x) for x in resp.index][::-1]
vals   = np.asarray(resp.values, dtype="int64")[::-1]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.barh(labels, vals, color="teal")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
ax.set_xlabel("Complaints"); ax.set_title("How complaints are resolved (all categories)")
finish(fig, "stage5_06a_resolution_overall.png",
       "Figure 6a. Most complaints close with an explanation (8.97M) or non-monetary relief such as a "
       "credit-file correction (5.39M). Monetary relief is comparatively rare (106k). 'In progress' "
       "(578k) are still open; '(missing)' (156k) lack a recorded outcome.")
resp

**Outcome mix by category** — as a 100%-stacked bar (each category's bar sums to 100%, so we
compare *composition* regardless of size).

In [ ]:
order = ["Closed with explanation","Closed with non-monetary relief","Closed with monetary relief",
         "Closed without relief","Closed with relief","Closed","In progress","Untimely response"]
rc = pd.crosstab(df["category"], df["Company response to consumer"], normalize="index")*100
rc = rc.reindex(index=LABELS, columns=[c for c in order if c in rc.columns])
fig, ax = plt.subplots(figsize=(9.5, 4.8))
left = np.zeros(len(rc)); pal = sns.color_palette("Spectral", len(rc.columns))
for col, color in zip(rc.columns, pal):
    ax.barh(rc.index, rc[col], left=left, label=col, color=color); left += rc[col].values
ax.set_xlabel("% of category"); ax.set_xlim(0,100); ax.set_title("Resolution mix by category (100%-stacked)")
ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
finish(fig, "stage5_06b_resolution_by_category.png",
       "Figure 6b. Credit reporting resolves overwhelmingly via non-monetary relief / explanation (file "
       "corrections), while debt collection and credit card show more monetary relief and more 'closed "
       "without relief' — outcomes clearly differ by category. Formal independence test: Stage 6.")
rc.round(1)

In [ ]:
tim = pd.crosstab(df["category"], df["Timely response?"], normalize="index")*100
tim = tim.reindex(LABELS)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(tim.index, tim["No"], color="indianred")
ax.set_ylabel("% answered LATE (untimely)"); ax.set_title("Untimely-response rate by category")
for i,v in enumerate(tim["No"]): ax.text(i, v, f"{v:.2f}%", ha="center", va="bottom", fontsize=9)
ax.margins(y=0.2)
finish(fig, "stage5_06c_timely_by_category.png",
       "Figure 6c. Timeliness is high everywhere (>96%), but debt collection is the worst: 3.49% of its "
       "complaints are answered late, versus just 0.15% for credit reporting.")

**Check whether carrying a narrative change the outcome?** For this, we compare the resolution mix for complaints **with**
vs **without** a published narrative.

In [ ]:
rn = pd.crosstab(df["has_narrative"], df["Company response to consumer"], normalize="index")*100
rn = rn.reindex(columns=[c for c in order if c in rn.columns])
rn.index = ["No narrative", "Has narrative"]
fig, ax = plt.subplots(figsize=(9.5, 3.4))
left = np.zeros(len(rn)); pal = sns.color_palette("Spectral", len(rn.columns))
for col, color in zip(rn.columns, pal):
    ax.barh(rn.index, rn[col], left=left, label=col, color=color); left += rn[col].values
ax.set_xlabel("% of group"); ax.set_xlim(0,100); ax.set_title("Resolution mix: narrative vs no narrative")
ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
finish(fig, "stage5_06d_resolution_by_narrative.png",
       "Figure 6d. Complaints that include a written narrative resolve in a broadly similar way to those "
       "without, with modest differences in the relief mix. Whether this difference is statistically "
       "meaningful is a Stage 6 question.")
rn.round(1)

## Summary

In [ ]:
lines = []
lines.append("STAGE 5 - DESCRIPTIVE TREND ANALYSIS - SUMMARY")
lines.append("="*47)
lines.append(f"Category shares (all 15,260,793 complaints): " +
             ", ".join(f"{c} {shares[c]:.1f}%" for c in LABELS))
lines.append("")
lines.append("VOLUME (all complaints, full counts incl. templated re-filings):")
lines.append(f"  Credit reporting per year: 2022={int(by_year.loc[2022,'Credit reporting']):,} -> "
             f"2025={int(by_year.loc[2025,'Credit reporting']):,} (~8x in 3 yrs). 2026 partial.")
lines.append("  Other three categories comparatively flat (mortgage declining).")
lines.append("")
lines.append("TEMPLATED SHARE (credit-reporting NARRATIVE subset only - different denominator):")
lines.append(f"  Within-year templated share: 2015={tmpl.loc[2015,'share_winyear_%']:.0f}% -> "
             f"2025={tmpl.loc[2025,'share_winyear_%']:.0f}%.")
lines.append(f"  Absolute templated CR narratives: 2015={int(tmpl.loc[2015,'templated_winyear']):,} -> "
             f"2025={int(tmpl.loc[2025,'templated_winyear']):,}.")
lines.append("  Within-year vs global definitions nearly identical -> growth is NOT a cross-year counting artifact.")
lines.append("  2026 excluded (narrative publication lags; only ~3,200 CR narratives so far).")
lines.append("")
lines.append("SEASONALITY: no strong fixed-month cycle; CR's Jan->Dec ramp is the steep growth leaking in.")
lines.append("  Formal trend/season decomposition deferred to Stage 6.")
lines.append("")
lines.append("COMPANIES (raw VOLUME, not rate - no market-share data):")
lines.append("  TransUnion 4.37M, Equifax 4.29M, Experian 3.88M dominate; next firm (Capital One) ~135k.")
lines.append("")
lines.append("RESOLUTION:")
lines.append(f"  Timely overall: Yes {(df['Timely response?'].astype('string').eq('Yes').mean()*100):.2f}%. "
             f"Worst category for lateness: debt collection ({tim.loc['Debt collection','No']:.2f}% late).")
lines.append("  CR resolves mostly via non-monetary relief/explanation; debt & credit card see more monetary relief.")
lines.append("")
lines.append("HONEST FLAGS:")
lines.append("  - Two denominators kept separate: volume/shares = ALL rows; templated share = CR NARRATIVE rows only.")
lines.append("  - 2026 partial across all charts; also narrative-incomplete (excluded from templated trend).")
lines.append("  - Company charts are volume not rate. Seasonality is descriptive only (Stage 6 = formal).")
lines.append("  - Narratives exist only from 2015, so the templated trend starts in 2015.")
summary = "\n".join(lines)
with open(os.path.join(REPORTS, "stage5_summary.txt"), "w", encoding="utf-8") as f:
    f.write(summary)
print(summary)